# Reusable template — unit-string catalog → regression price
## Clone of `LaptopPrice`

Swap the CSV and the column lists. Keep the cardinality rule, the train-only target encoding, and the MAE-vs-baseline habit.


**Short name:** `LaptopPrice` template

Typical swaps: used cars (engine CC, mileage), phones (RAM / storage), rents (sqft + BHK), cameras (MP + crop), hotel ADR (room class).


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# ---- edit these ----
DATA_PATH = "data/laptop_price.csv"
TARGET = "Price"
UNIT_COLS = ["ram_gb", "ssd", "hdd", "graphic_card_gb"]   # strip this unit
UNIT_TOKEN = " GB"
COMPOSITE = ("total_storage", ["ssd", "hdd"])              # or None
SENTINEL_COL = "processor_gnrtn"                           # or None
SENTINEL_FROM, SENTINEL_TO = "Not Available", "0"
CARD_CUTOFF = 5
TEST_SIZE = 0.2
RANDOM_STATE = 42


In [ ]:
df = pd.read_csv(DATA_PATH)
for col in UNIT_COLS:
    df[col] = df[col].astype(str).str.replace(UNIT_TOKEN, "", regex=False)
    df[col] = df[col].str.extract(r"(\d+)", expand=False).astype(float)

if COMPOSITE:
    name, parts = COMPOSITE
    df[name] = df[parts].sum(axis=1)
    df = df.drop(columns=parts)

# add project-specific artifact strips here (bit, stars, 'th', currency symbols…)

y = df[TARGET]
X = df.drop(columns=[TARGET])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
low = [c for c in cat_cols if X[c].nunique() < CARD_CUTOFF]
high = [c for c in cat_cols if X[c].nunique() >= CARD_CUTOFF]

for col in high:
    means = pd.concat([X_train[col], y_train], axis=1).groupby(col)[TARGET].mean()
    X_train[col] = X_train[col].map(means)
    X_test[col] = X_test[col].map(means).fillna(y_train.mean())
for col in low:
    dtr = pd.get_dummies(X_train[col], prefix=col, drop_first=True)
    dte = pd.get_dummies(X_test[col], prefix=col, drop_first=True)
    dte = dte.reindex(columns=dtr.columns, fill_value=0)
    X_train = pd.concat([X_train.drop(columns=[col]), dtr], axis=1)
    X_test = pd.concat([X_test.drop(columns=[col]), dte], axis=1)

# leftover object cols? drop or encode
obj = X_train.select_dtypes(include=["object"]).columns.tolist()
X_train = X_train.drop(columns=obj)
X_test = X_test.drop(columns=obj)

model = RandomForestRegressor(random_state=RANDOM_STATE)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("MAE", mean_absolute_error(y_test, pred))
print("R2 ", r2_score(y_test, pred))
print("baseline MAE", mean_absolute_error(y_test, np.full_like(y_test, y_train.mean(), dtype=float)))
print(pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(8))
